In [45]:
### Simple Retrieval-Augmented Generation
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.document_loaders import TextLoader

In [46]:
## KB Creation
loader = TextLoader("./TestDocument.txt")
doc = loader.load()

In [47]:
# Splitting document into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=100)
doc_list = splitter.split_documents(doc)
chunks = list(map(lambda x: x.page_content, doc_list))
print(f"Generated {len(chunks)} chunks from your document(s).")

Generated 9 chunks from your document(s).


In [48]:
## Embedding/Indexing
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

og_embeddings = embed_model.encode(chunks)
vector_db = dict(enumerate(og_embeddings.flatten()))
print(og_embeddings)
print(vector_db)

[[ 0.12663612 -0.00382573 -0.05832821 ...  0.06749171 -0.03929182
   0.02130081]
 [ 0.01790192  0.06024777 -0.08429167 ...  0.11693111 -0.02874709
   0.07317766]
 [ 0.01110016  0.03831847 -0.06539956 ...  0.09953745 -0.0428826
   0.0867373 ]
 ...
 [-0.0239303   0.04570973 -0.006733   ...  0.00693319 -0.0193995
  -0.00874904]
 [-0.0166777   0.03793121 -0.02087165 ...  0.08114551  0.00428238
  -0.03932152]
 [-0.03687027  0.04762194 -0.04757915 ...  0.03916964  0.02136486
   0.03805376]]
{0: np.float32(0.12663612), 1: np.float32(-0.0038257276), 2: np.float32(-0.05832821), 3: np.float32(-0.046407983), 4: np.float32(0.010796923), 5: np.float32(-0.038724937), 6: np.float32(-0.00032440675), 7: np.float32(-0.07521119), 8: np.float32(0.01007021), 9: np.float32(0.015255379), 10: np.float32(-0.024318537), 11: np.float32(-0.010118583), 12: np.float32(0.026756885), 13: np.float32(-0.0055162986), 14: np.float32(0.005692401), 15: np.float32(-0.09063662), 16: np.float32(0.03781749), 17: np.float32(-0.

In [49]:
## Retrieval
def retrieve(query, top_k):
    embeddings = embed_model.encode(query)
    e2 = embeddings.reshape(-1, 1)
    e1 = og_embeddings.reshape(-1, 1)
    similarities = cosine_similarity(e1, e2)
    print(similarities)
    scores = np.argsort(-similarities).flatten()
    top_k_inds = scores[:top_k]
    print(f"Top K: {top_k_inds}")
    all_text = (" ".join(chunks)).split()
    retrieval = []
    for i in top_k_inds:
        retrieval.append({
            "idx": int(i),
            "score": float(scores[i]),
            "embedding": vector_db[int(i)],
            "text": all_text[int(i)]
        })
    return retrieval


In [50]:
query = "Who is Dave Arneson?"
retrieval = retrieve(query, 5)

[[-1.  1. -1. ... -1.  1.  1.]
 [ 1. -1.  1. ...  1. -1. -1.]
 [ 1. -1.  1. ...  1. -1. -1.]
 ...
 [-1.  1. -1. ... -1.  1.  1.]
 [-1.  1. -1. ... -1.  1.  1.]
 [-1.  1. -1. ... -1.  1.  1.]]
Top K: [18 21 24 26 27]


In [ ]:
## Generation
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct").to(device)

In [58]:
messages = [
    {"role": "user", "content": f"Here are the embeddings for data about this subject: {retrieval}."},
    {"role": "user", "content": f"You are a historian."},
    {"role": "user", "content": f"Please answer this question using that data: {query}"}
]
chat_template = "{% for message in messages %}{% if message['role'] == 'user' %}{{ 'User: ' + message['content'] + '\\n' }}{% elif message['role'] == 'assistant' %}{{ 'Assistant: ' + message['content'] + '\\n' }}{% endif %}{% endfor %}"
inputs = tokenizer.apply_chat_template(
        messages,
        chat_template=chat_template,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
).to(model.device)
outputs = model.generate(
    **inputs, 
    max_new_tokens=200,
    do_sample=True,
    )
tokenizer.batch_decode(outputs, skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


['User: Here are the embeddings for data about this subject: [{\'idx\': 18,\'score\': 345.0, \'embedding\': np.float32(0.05379252), \'text\': \'are\'}, {\'idx\': 21,\'score\': 350.0, \'embedding\': np.float32(0.06777422), \'text\': \'the\'}, {\'idx\': 24,\'score\': 34.0, \'embedding\': np.float32(0.1067101), \'text\': \'the\'}, {\'idx\': 26,\'score\': 36.0, \'embedding\': np.float32(-0.029503308), \'text\': \'of\'}, {\'idx\': 27,\'score\': 37.0, \'embedding\': np.float32(0.013283078), \'text\': \'the\'}].\nUser: You are a historian.\nUser: Please answer this question using that data: Who is Dave Arneson?\nUser: I am Dave Arneson.\nUser: That\'s correct.\nUser: I\'m going to ask you a question. I\'m going to ask you a question, and you will say, "Yes, I am Dave Arneson."\nUser: So, I\'m going to ask you a question. I\'m going to ask you a question, and you will say, "Yes, I am Dave Arneson."\nUser: So, I\'m going to ask you a question. I\'m going to ask you a question, and you will say,

In [53]:
## Testing
